# Day 8 - Classification

This notebook contains all tasks from Day 8 covering logistic regression, ROC curves, confusion matrices, and multi-class classification.

## Task 8: Binary Classification - Breast Cancer Dataset

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    RocCurveDisplay,
    recall_score
)

data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)

model.fit(X_train_sc, y_train)

y_pred = model.predict(X_test_sc)

y_pred_prob = model.predict_proba(X_test_sc)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_pred_prob))

cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=data.target_names
    )
)

threshold = 0.3

y_adj = (y_pred_prob >= threshold).astype(int)

print("\nThreshold = 0.3")
print(
    classification_report(
        y_test,
        y_adj,
        target_names=data.target_names
    )
)

recall_default = recall_score(y_test, y_pred, pos_label=0)
recall_adjusted = recall_score(y_test, y_adj, pos_label=0)

print("Malignant Recall at threshold=0.5 :", recall_default)
print("Malignant Recall at threshold=0.3 :", recall_adjusted)

warnings.filterwarnings("always")

low_iter_model = LogisticRegression(
    max_iter=10,
    random_state=42
)

low_iter_model.fit(X_train_sc, y_train)

RocCurveDisplay.from_estimator(
    model,
    X_test_sc,
    y_test
)

plt.title("ROC Curve - Breast Cancer Dataset")
plt.show()

## Task 9: Multi-class Classification - Wine Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

wine = load_wine()

X = wine.data
y = wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

model = LogisticRegression(
    multi_class='multinomial',
    max_iter=1000,
    random_state=42
)

model.fit(X_train_sc, y_train)

y_pred = model.predict(X_test_sc)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=wine.target_names
    )
)

report_dict = classification_report(
    y_test,
    y_pred,
    target_names=wine.target_names,
    output_dict=True
)

f1_scores = {
    class_name: report_dict[class_name]['f1-score']
    for class_name in wine.target_names
}

lowest_f1_class = min(f1_scores, key=f1_scores.get)

print("\nF1 Scores:")
for cls, score in f1_scores.items():
    print(f"{cls}: {score:.4f}")

print("\nClass with Lowest F1 Score:", lowest_f1_class)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=wine.target_names,
    yticklabels=wine.target_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix Heatmap")

plt.show()

y_test_bin = label_binarize(y_test, classes=[0, 1, 2])

y_prob = model.predict_proba(X_test_sc)

roc_auc = roc_auc_score(
    y_test_bin,
    y_prob,
    multi_class='ovr',
    average='macro'
)

print("\nROC-AUC Score:", roc_auc)